In [4]:
from transformers import EsmTokenizer, EsmForMaskedLM

model_name = 'facebook/esm1v_t33_650M_UR90S_1'
tokenizer = EsmTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name)

Some weights of EsmForMaskedLM were not initialized from the model checkpoint at facebook/esm1v_t33_650M_UR90S_1 and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
import torch

def calculate_pppl(model, tokenizer, sequence):
    """Calculates the pseudo-perplexity score of a sequence"""
    token_ids = tokenizer.encode(sequence, return_tensors='pt')
    input_length = token_ids.size(1)
    log_likelihood = 0.0

    for i in range(1, input_length - 1):
        # Create a copy of the token IDs
        masked_token_ids = token_ids.clone()
        # Mask a token that we will try to predict back
        masked_token_ids[0, i] = tokenizer.mask_token_id

        with torch.no_grad():
            output = model(masked_token_ids)
            logit_prob = torch.nn.functional.log_softmax(output.logits, dim=-1)
        
        log_likelihood += logit_prob[0, i, token_ids[0, i]]

    # Calculate the average log likelihood per token
    avg_log_likelihood = log_likelihood / (input_length - 2)

    # Compute and return the pseudo-perplexity
    pppl = torch.exp(-avg_log_likelihood)
    return pppl.item()

In [6]:
# Calculate for spycatcher
import torch

SPYCATCHER = 'VDTLSGLSSEQGPSGDMTIEEDSATHIKFSKRDEDGKELAGATMELRDSSGKTISTWISDGQVKDFYLYPGKYTFVETAAPDGYEVATAITFTVNEQGQVTVNGKATKGDAHI'

calculate_pppl(model, tokenizer, SPYCATCHER)

8.673417091369629

In [4]:
from datasets import load_from_disk

test_split = load_from_disk('../../data/processed/mlm_tokenized')['test']
test_split

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 14081
})

In [5]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

def batch_evaluate_pppl(model, tokenizer, data_loader, device):
    model.eval()
    total_log_likelihood = 0.0
    total_tokens = 0
    pad_token_id = tokenizer.pad_token_id
    mask_token_id = tokenizer.mask_token_id

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch[0].to(device)  # Shape: (batch_size, seq_len)
            batch_size, seq_len = input_ids.shape

            # Padding mask (True for non-padding tokens)
            if pad_token_id is not None:
                padding_mask = (input_ids != pad_token_id)
            else:
                padding_mask = torch.ones_like(input_ids, dtype=torch.bool)

            total_tokens += padding_mask.sum().item()

            for i in range(seq_len):
                current_padding_mask = padding_mask[:, i]

                # Skip if all tokens at position i are padding
                if not torch.any(current_padding_mask):
                    continue

                # Mask the i-th token in all sequences
                masked_batch = input_ids.clone()
                masked_batch[:, i] = mask_token_id

                # Forward pass
                outputs = model(masked_batch)
                logits = outputs.logits
                log_probs = F.log_softmax(logits, dim=-1)

                # Get true tokens and their log-probabilities
                true_tokens = input_ids[:, i]
                log_probs_at_i = log_probs[:, i, :]
                batch_log_likelihood = log_probs_at_i.gather(dim=1, index=true_tokens.unsqueeze(1)).squeeze(1)

                # Apply padding mask
                masked_log_likelihood = batch_log_likelihood * current_padding_mask.float()

                # Accumulate log-likelihood
                total_log_likelihood += masked_log_likelihood.sum()

    # Compute final pseudo-perplexity
    avg_log_likelihood = total_log_likelihood / total_tokens
    pppl = torch.exp(-avg_log_likelihood)
    return pppl.item()

In [6]:
token_ids = [torch.tensor(ids) for ids in test_split['input_ids'][:50]]

# Convert to a batched tensor and create a DataLoader
all_input_ids = torch.nn.utils.rnn.pad_sequence(
    sequences=token_ids, 
    batch_first=True, 
    padding_value=tokenizer.pad_token_id
)

dataset = TensorDataset(all_input_ids)
data_loader = DataLoader(dataset, batch_size=8, shuffle=False)

# Evaluate
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

pppl = batch_evaluate_pppl(model, tokenizer, data_loader, device)
print(f"Pseudo-Perplexity: {pppl:.4f}")

Pseudo-Perplexity: 8.6121


In [1]:
import pandas as pd

# Define model and dataset names
model_names = ['ModelA', 'ModelB', 'ModelC']
dataset_names = ['Dataset1', 'Dataset2', 'Dataset3']

# Create empty DataFrame
df = pd.DataFrame(index=model_names, columns=dataset_names)

# Insert values procedurally
df.loc['ModelA', 'Dataset1'] = 0.87
df.loc['ModelB', 'Dataset2'] = 0.92

In [2]:
df

,Dataset1,Dataset2,Dataset3
ModelA,0.87,NaN,NaN
ModelB,NaN,0.92,NaN
ModelC,NaN,NaN,NaN


In [7]:
import pandas as pd

pd.read_csv('../../data/processed/pppl_matrix.csv')

,fbp-79k,scramm-480k,adhesin-3M
0,10.596148,NaN,NaN
1,12.464245,NaN,NaN
2,10.402374,NaN,NaN
